In [1]:
import pandas as pd
import requests
import re

In [2]:
url = "https://users.stat.ufl.edu/~winner/data/armada.dat"

In [3]:
text = requests.get(url).text.splitlines()  #ChatGPT (2025) support on this process. Struggled to upload.

rows = []

for line in text:
    parts = re.split(r"\s+(?=\d)", line, maxsplit=1)
    name = parts[0].strip()
    nums = parts[1].split()
    rows.append([name] + nums)

In [4]:
data = pd.DataFrame(rows)

In [14]:
df = data.copy()
df

,0,1,2,3,4,5,6,7
0,Bantam,1601,6,3,0,2,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.5,0,0
5,Ilha das Naus,1615,3,5,0,0.6,0,-1
6,Jask,1620,4,0,4,1,0,0
7,Hormuz,1622,6,0,5,1.2,0,-1
8,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
9,Hormuz,1625,8,4,4,1,0,0


In [6]:
df.columns = ["Battle", "Year", "Portuguese_ships", "Dutch_ships", "English_ships", "Ratio_Portuguese_Dutch_British", "Spanish_Involvement", "Portuguese_Outcome"]

In [7]:
df.head()

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_Portuguese_Dutch_British,Spanish_Involvement,Portuguese_Outcome
0,Bantam,1601,6,3,0,2,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.5,0,0


In [8]:
df.shape

(28, 8)

In [9]:
df.isnull().sum()

Battle                            0
Year                              0
Portuguese_ships                  0
Dutch_ships                       0
English_ships                     0
Ratio_Portuguese_Dutch_British    0
Spanish_Involvement               0
Portuguese_Outcome                0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df[["Portuguese_ships", "Dutch_ships", "English_ships", "Spanish_Involvement"]]
y = df["Portuguese_Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3)

model = SVC(kernel="linear")
model.fit(X_train, y_train)

preds = model.predict(X_test)

svm_acc = accuracy_score(y_test, preds)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

[[2 1 0]
 [3 1 0]
 [0 2 0]]
              precision    recall  f1-score   support

          -1       0.40      0.67      0.50         3
           0       0.25      0.25      0.25         4
           1       0.00      0.00      0.00         2

    accuracy                           0.33         9
   macro avg       0.22      0.31      0.25         9
weighted avg       0.24      0.33      0.28         9



/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [11]:
#Now I will try a Decision Tree Classifier

from sklearn.tree import DecisionTreeClassifier

X = df[["Portuguese_ships", "Dutch_ships", "English_ships", "Spanish_Involvement"]]
y = df["Portuguese_Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_preds = dt.predict(X_test)

dt_acc = accuracy_score(y_test, dt_preds)
print(confusion_matrix(y_test, dt_preds))
print(classification_report(y_test, dt_preds))


[[2 0 0]
 [3 1 1]
 [2 0 0]]
              precision    recall  f1-score   support

          -1       0.29      1.00      0.44         2
           0       1.00      0.20      0.33         5
           1       0.00      0.00      0.00         2

    accuracy                           0.33         9
   macro avg       0.43      0.40      0.26         9
weighted avg       0.62      0.33      0.28         9



In [12]:
#Now I will use a random forest classifier

from sklearn.ensemble import RandomForestClassifier 

rf = RandomForestClassifier(
    n_estimators=200, 
    random_state=42,
    max_depth=None,
    min_samples_leaf=2)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds))


[[2 0 0]
 [2 3 0]
 [1 1 0]]
              precision    recall  f1-score   support

          -1       0.40      1.00      0.57         2
           0       0.75      0.60      0.67         5
           1       0.00      0.00      0.00         2

    accuracy                           0.56         9
   macro avg       0.38      0.53      0.41         9
weighted avg       0.51      0.56      0.50         9



/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [13]:
results = pd.DataFrame({ "Model": ["SVM", "Decision Tree", "Random Forest"],
                        "Accuracy": [svm_acc, dt_acc, rf_acc]})

print(results)

           Model  Accuracy
0            SVM  0.333333
1  Decision Tree  0.333333
2  Random Forest  0.555556


Comparing the accuracy of these 3 models I can see that random forest performed best on this data set. I needed to adjust my test size to be large enough to incorporate all 3 classes and this lowered the accuracy of the SVM model when I adjusted it to the lowest accuracy rate of 22%. The Random Forest classified the draw outcomes the strongest. As this data set is only 28 samples, it is a bit tight to work with. 